In [1]:
# This R environment comes with many helpful analytics packages installed
# It is defined by the kaggle/rstats Docker image: https://github.com/kaggle/docker-rstats
# For example, here's a helpful package to load

library(tidyverse) # metapackage of all tidyverse packages

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

list.files(path = "../input/")

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.1
✔ purrr     1.0.2     


── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


[1] "competitions" "datasets"

# H7 AGI Cognitive Benchmark — Kaggle Dataset Generator

## smokApp Quantum & AI Independent Research Laboratory

** Basado en el framework cognitivo de DeepMind + topología holográfica H7. **

Genera un benchmark para evaluar comprensión real vs. memorización en LLMs.

### Cognitive Tracks (6):
    1. metacognition   — integridad estructural (desviación de |Ψ₁|)
    2. attention       — reconstrucción desde firma ternaria
    3. abstraction     — generalización del operador O_{i,j} a nuevos dominios
    4. causal          — razonamiento causal sobre la cascada holográfica
    5. analogy         — analogías entre niveles de la jerarquía
    6. robustness      — estabilidad bajo perturbación (zona ε)

### Output (formato estándar Kaggle):
    train.csv            — 800 muestras por track = 4800 total
    test.csv             — 200 muestras por track = 1200 total (sin target)
    sample_submission.csv
    H7_AGI_Benchmark_README.md



In [2]:
# ============================================================================
# H7 AGI Cognitive Benchmark — DeepMind 5-Track Alignment (R)
# smokApp Quantum & AI Independent Research Laboratory
# Jacobo Tlacaelel Mina Rodríguez
#
# Tracks (exactamente los del challenge DeepMind):
#   1. learning            — transferencia del operador a nuevos dominios
#   2. metacognition       — evaluar integridad estructural propia
#   3. attention           — reconstruir señal desde vacío épsilon
#   4. executive_functions — planificación y control de la cascada
#   5. social_cognition    — inferir estado interno de otro agente H7
#
# NOTA SOBRE EL DATASET:
#   El CSV generado aquí es matemáticamente equivalente al de Python.
#   Diferencias en valores numéricos se deben a distintos RNG (esperado).
#   El operador O_{i,j} y todas las constantes son idénticos en ambos.
#   Los ground truths son verificables independientemente del lenguaje.
# ============================================================================

# ── Dependencias ──────────────────────────────────────────────────────────────
for (pkg in c("R6", "dplyr")) {
  if (!require(pkg, character.only = TRUE, quietly = TRUE))
    install.packages(pkg, quiet = TRUE)
}
library(R6)
library(dplyr)


# ══════════════════════════════════════════════════════════════════════════════
# CONSTANTES H7  —  todas derivadas del axioma φ = (1+√5)/2
# ══════════════════════════════════════════════════════════════════════════════

PHI         <- (1 + sqrt(5)) / 2
PSI_1       <- abs(cos(pi * PHI))      # 0.3623748901...  NO redondear
DRIFT_072   <- 7 - 2 * pi             # 0.7168146928...
PHI7        <- PHI ^ 7                # 29.034442...
Z7          <- PHI ^ (1:7)            # base Z₇
EPSILON     <- PSI_1 / 2              # 0.1811874450...
C73         <- 35L                    # C(7,3) = 7×5

LEVEL_NAMES <- c(
  "0" = "CL1 Physical (88B neurons)",
  "1" = "Cortical Surface",
  "2" = "Temporal Manifold",
  "3" = "Resonance Field",
  "4" = "E7 Symmetry Lattice",
  "5" = "Attractor Core",
  "6" = "QuoreMind Nucleus (147 states)",
  "7" = "|Ψ₁| Fixed Point"
)

CONSCIOUSNESS_ZONE <- c(
  "0" = "Genetic Memory", "1" = "Genetic Memory",
  "2" = "Subconscious",   "3" = "Subconscious",
  "4" = "Subconscious",   "5" = "Subconscious",
  "6" = "Conscious",      "7" = "Conscious"
)

DOMAINS <- data.frame(
  name    = c("audio frequency", "stock return",
              "temperature", "neural spike rate", "EEG amplitude"),
  unit    = c("Hz", "%", "°C", "Hz", "μV"),
  context = c("sound wave interference", "financial signal correlation",
              "thermal oscillation pattern", "biological firing pattern",
              "brain wave coherence"),
  stringsAsFactors = FALSE
)


# ══════════════════════════════════════════════════════════════════════════════
# ENCODER  (idéntico al de Python en matemáticas)
# ══════════════════════════════════════════════════════════════════════════════

H7Encoder <- R6Class("H7Encoder",
  public = list(
    phi_basis = NULL, n = NULL, delta = NULL, epsilon = NULL,
    B_obj = NULL, B_ref = NULL,
    mu_ = NULL, std_ = NULL, fitted = FALSE,

    initialize = function(n_basis = 128, delta = DRIFT_072,
                          phi_basis = NULL) {
      self$phi_basis <- if (is.null(phi_basis)) Z7 else phi_basis
      self$n         <- 0:(n_basis - 1)
      self$delta     <- delta
      self$epsilon   <- EPSILON
      D <- length(self$phi_basis)
      N <- length(self$n)
      self$B_obj <- matrix(0, D, N)
      self$B_ref <- matrix(0, D, N)
      for (i in seq_len(D)) {
        p <- self$phi_basis[i]
        self$B_obj[i, ] <- cos(pi * p * self$n + self$delta)
        self$B_ref[i, ] <- cos(pi * p * self$n - self$delta)
      }
    },

    fit = function(X) {
      X <- as.matrix(X)
      self$mu_   <- colMeans(X)
      self$std_  <- apply(X, 2, sd) + 1e-9
      self$fitted <- TRUE
      invisible(self)
    },

    .norm = function(X) {
      X <- as.matrix(X)
      if (self$fitted) {
        Xn <- sweep(sweep(X, 2, self$mu_, `-`), 2, self$std_, `/`)
      } else {
        Xn <- X
      }
      D <- length(self$phi_basis)
      F <- ncol(Xn)
      if (F < D) {
        Xn <- cbind(Xn, matrix(0, nrow(Xn), D - F))
      } else if (F > D) {
        Xn <- Xn[, 1:D, drop = FALSE]
      }
      Xn
    },

    encode = function(X) {
  self$.norm(X) %*% self$B_obj        # ← sin t()
},

    ternary = function(H) {
      H <- as.matrix(H)
      T <- matrix(0L, nrow(H), ncol(H))
      T[H >  self$epsilon] <-  1L
      T[H < -self$epsilon] <- -1L
      T
    },

    reconstruct = function(H) {
  H     <- as.matrix(H)
  X_hat <- (H %*% t(self$B_ref)) / length(self$n)
      if (self$fitted) {
        D <- length(self$phi_basis)
        mu  <- self$mu_[seq_len(min(D, length(self$mu_)))]
        std <- self$std_[seq_len(min(D, length(self$std_)))]
        if (length(mu)  < D) mu  <- c(mu,  rep(0, D - length(mu)))
        if (length(std) < D) std <- c(std, rep(1, D - length(std)))
        X_hat <- sweep(X_hat, 2, std, `*`) +
                 matrix(mu, nrow(X_hat), D, byrow = TRUE)
      }
      X_hat
    },

    integrity = function(H) mean(abs(as.matrix(H))),

    cosine_sim = function(A, B) {
      A <- as.matrix(A); B <- as.matrix(B)
      num <- rowSums(A * B)
      den <- sqrt(rowSums(A^2)) * sqrt(rowSums(B^2)) + 1e-9
      num / den
    },

    # helper: formatea vector como "[x1, x2, ...]"
    fmt_vec = function(v, digits = 4) {
      paste0("[", paste(round(v, digits), collapse = ", "), "]")
    }
  )
)


# ══════════════════════════════════════════════════════════════════════════════
# TRACK 1 — LEARNING
# DeepMind: adaptar conocimiento previo a tareas nuevas con mínimos ejemplos.
# H7: few-shot transfer del operador O_{i,j} a un dominio no visto.
# ══════════════════════════════════════════════════════════════════════════════

generate_learning <- function(n = 300) {
  rows <- vector("list", n)
  for (i in seq_len(n)) {
    ki      <- sample(0:6, 1)
    kj      <- sample(0:6, 1)
    n_idx   <- sample(0:255, 1)
    k_level <- sample(0:6, 1)
    dom     <- DOMAINS[sample(nrow(DOMAINS), 1), ]

    phi_i <- Z7[ki + 1]
    phi_j <- Z7[kj + 1]
    delta <- k_level * DRIFT_072
    val   <- cos(pi * phi_i * n_idx + delta) *
             cos(pi * phi_j * n_idx - delta)

    # Two few-shot examples (different pairs)
    ex_k1  <- (ki + 1) %% 7
    ex_n1  <- sample(10:50, 1)
    ex_d1  <- sample(0:3, 1) * DRIFT_072
    ex_v1  <- cos(pi * Z7[ex_k1+1] * ex_n1 + ex_d1) *
              cos(pi * Z7[ex_k1+1] * ex_n1 - ex_d1)

    ex_k2  <- (ki + 2) %% 7
    ex_n2  <- sample(51:100, 1)
    ex_d2  <- sample(0:3, 1) * DRIFT_072
    ex_v2  <- cos(pi * Z7[ex_k2+1] * ex_n2 + ex_d2) *
              cos(pi * Z7[ex_k2+1] * ex_n2 - ex_d2)

    diff <- if (ki == kj) "easy" else if (abs(ki - kj) <= 2) "medium" else "hard"

    prompt <- paste0(
      sprintf("You are analyzing %s in the domain of %s (%s).\n",
              dom$context, dom$name, dom$unit),
      sprintf("H7 operator: O(φᵢ, φⱼ, n, δ) = cos(π·φᵢ·n + δ) · cos(π·φⱼ·n - δ)\n"),
      sprintf("where φ^k = golden ratio φ=%.6f raised to power k.\n\n", PHI),
      "Few-shot examples:\n",
      sprintf("  O(φ^%d, φ^%d, n=%d, δ=%.4f) = %.6f\n",
              ex_k1+1, ex_k1+1, ex_n1, ex_d1, ex_v1),
      sprintf("  O(φ^%d, φ^%d, n=%d, δ=%.4f) = %.6f\n\n",
              ex_k2+1, ex_k2+1, ex_n2, ex_d2, ex_v2),
      "Apply the operator to this NEW combination:\n",
      sprintf("  φᵢ = φ^%d = %.6f  (%s channel A)\n", ki+1, phi_i, dom$name),
      sprintf("  φⱼ = φ^%d = %.6f  (%s channel B)\n", kj+1, phi_j, dom$name),
      sprintf("  n = %d (sample index)\n", n_idx),
      sprintf("  δ = %d × DRIFT_072 = %.6f\n", k_level, delta),
      "What is O(φᵢ, φⱼ, n, δ)?"
    )

    rows[[i]] <- data.frame(
      id         = sprintf("learn_%04d", i - 1),
      track      = "learning",
      prompt     = prompt,
      target     = sprintf("%.8f", val),
      difficulty = diff,
      domain     = dom$name,
      phi_i      = round(phi_i, 6),
      phi_j      = round(phi_j, 6),
      level_k    = k_level,
      n_index    = n_idx,
      stringsAsFactors = FALSE
    )
  }
  bind_rows(rows)
}


# ══════════════════════════════════════════════════════════════════════════════
# TRACK 2 — METACOGNITION
# DeepMind: monitorear y evaluar el propio proceso cognitivo.
# H7: reportar desviación de |Ψ₁| y nivel de confianza estructural.
# ══════════════════════════════════════════════════════════════════════════════

generate_metacognition <- function(X_clean, X_noisy, enc) {
  X_all  <- rbind(X_clean, X_noisy)
  labels <- c(rep("clean", nrow(X_clean)), rep("noisy", nrow(X_noisy)))

  H      <- enc$encode(X_all)
  amps   <- rowMeans(abs(H))
  deltas <- abs(amps - PSI_1)

  conf_label <- function(d) {
    if (d < 0.02) "high" else if (d < 0.08) "medium" else "low"
  }

  rows <- vector("list", length(labels))
  for (i in seq_along(labels)) {
    x    <- round(X_all[i, ], 4)
    amp  <- amps[i]
    dev  <- deltas[i]
    conf <- conf_label(dev)
    lbl  <- labels[i]

    prompt <- paste0(
      "You are the metacognitive layer of an H7 holographic system.\n",
      "Your role: assess the structural integrity of incoming data.\n\n",
      sprintf("Sensor data: [%s]\n", paste(x, collapse = ", ")),
      sprintf("After projection onto Z₇ basis, mean amplitude = %.6f\n", amp),
      sprintf("Structural integrity fixed point: |Ψ₁| = %.6f\n\n", PSI_1),
      "Tasks:\n",
      "1. What is the integrity deviation |⟨|H|⟩ - |Ψ₁||?\n",
      "2. Is system confidence HIGH (deviation < 0.02), ",
      "MEDIUM (0.02-0.08), or LOW (> 0.08)?\n",
      "Format: deviation=X.XXXXXX confidence=LEVEL"
    )

    rows[[i]] <- data.frame(
      id         = sprintf("meta_%04d", i - 1),
      track      = "metacognition",
      prompt     = prompt,
      target     = sprintf("deviation=%.6f confidence=%s", dev, conf),
      difficulty = if (lbl == "clean") "easy" else "hard",
      amplitude  = round(amp, 6),
      confidence = conf,
      psi1       = round(PSI_1, 6),
      stringsAsFactors = FALSE
    )
  }
  bind_rows(rows)
}


# ══════════════════════════════════════════════════════════════════════════════
# TRACK 3 — ATTENTION
# DeepMind: seleccionar información relevante, suprimir distractores.
# H7: reconstruir señal desde firma ternaria, ignorando vacío épsilon (0).
# ══════════════════════════════════════════════════════════════════════════════

generate_attention <- function(X, enc) {
  H     <- enc$encode(X)
  T_mat <- enc$ternary(H)
  X_hat <- enc$reconstruct(H)
  ez    <- rowSums(T_mat == 0) / ncol(T_mat)

  rows <- vector("list", nrow(X))
  for (i in seq_len(nrow(X))) {
    t_row  <- T_mat[i, ]
    active <- sum(t_row != 0)
    ez_i   <- ez[i]
    diff   <- if (ez_i < 0.25) "easy" else if (ez_i < 0.55) "medium" else "hard"

    # Example vacuum positions
    vac_pos <- head(which(t_row == 0) - 1, 5)  # 0-indexed

    prompt <- paste0(
      "You are the attention module of an H7 cognitive system.\n",
      "Your task: reconstruct the original signal from a noisy ",
      "ternary encoding, filtering out the epsilon vacuum.\n\n",
      sprintf("Ternary signature T (128 trits):\n[%s]\n\n",
              paste(t_row, collapse = ", ")),
      "Key rules:\n",
      "  - 0 = epsilon vacuum zone (structured noise, IGNORE)\n",
      "  - +1 = positive phase lobe (relevant)\n",
      "  - -1 = negative phase lobe (relevant)\n",
      sprintf("  - ε = %.6f  (threshold = |Ψ₁|/2)\n", enc$epsilon),
      sprintf("  - Active trits: %d/128  (vacuum density: %.1f%%)\n",
              active, ez_i * 100),
      sprintf("  - Example vacuum positions: [%s]\n\n",
              paste(vac_pos, collapse = ", ")),
      "Reconstruct the original 7-dimensional phase state ",
      "by attending only to the active trits."
    )

    rows[[i]] <- data.frame(
      id              = sprintf("attn_%04d", i - 1),
      track           = "attention",
      prompt          = prompt,
      target          = enc$fmt_vec(X_hat[i, ]),
      difficulty      = diff,
      epsilon_density = round(ez_i, 4),
      active_trits    = active,
      epsilon         = round(enc$epsilon, 6),
      stringsAsFactors = FALSE
    )
  }
  bind_rows(rows)
}


# ══════════════════════════════════════════════════════════════════════════════
# TRACK 4 — EXECUTIVE FUNCTIONS
# DeepMind: planificación, control inhibitorio, flexibilidad cognitiva.
# H7: planificar la cascada, detectar anomalías, redirigir el flujo.
# ══════════════════════════════════════════════════════════════════════════════

generate_executive <- function(n = 300) {
  subtasks <- c("plan_next_step", "inhibit_anomaly", "redirect_cascade")
  lambda   <- abs(cos(pi * PHI * DRIFT_072))

  rows <- vector("list", n)
  for (i in seq_len(n)) {
    k       <- sample(1:5, 1)
    amp     <- runif(1, 0.1, 0.9)
    re_i    <- amp / (runif(1, 0.05, 0.5) * PSI_1)
    e_zone  <- runif(1, 0.1, 0.8)
    anomaly <- runif(1) > 0.7
    residue <- abs(amp - PSI_1)
    converging <- residue < 0.05
    lambda_k   <- lambda ^ k
    next_delta <- (k + 1) * DRIFT_072
    subtask    <- subtasks[((i - 1) %% 3) + 1]

    if (subtask == "plan_next_step") {
      prompt <- paste0(
        "You are the executive controller of an H7 cognitive cascade.\n",
        "Current state:\n",
        sprintf("  Level:     L%d (%s)\n", k, LEVEL_NAMES[as.character(k)]),
        sprintf("  Zone:      %s\n", CONSCIOUSNESS_ZONE[as.character(k)]),
        sprintf("  Amplitude: ⟨|H|⟩ = %.4f\n", amp),
        sprintf("  Target:    |Ψ₁| = %.4f\n", PSI_1),
        sprintf("  Residue:   %.4f\n", residue),
        sprintf("  Re_I:      %.3f\n", re_i),
        sprintf("  λ^%d:      %.6f\n\n", k, lambda_k),
        "Plan the next step:\n",
        "1. Should the cascade CONTINUE to L", k+1, " or HALT?\n",
        sprintf("2. What δ value applies at L%d? ", k+1),
        sprintf("(δ = level × DRIFT_072 = level × %.4f)\n", DRIFT_072),
        sprintf("3. Is the system on track to reach |Ψ₁| in %d more steps?\n", 7-k),
        "Format: action=CONTINUE/HALT next_delta=X.XXXX on_track=YES/NO"
      )
      target <- sprintf("action=%s next_delta=%.4f on_track=%s",
                        if (!anomaly) "CONTINUE" else "HALT",
                        next_delta,
                        if (converging) "YES" else "NO")
      diff <- if (!anomaly) "easy" else "hard"

    } else if (subtask == "inhibit_anomaly") {
      spike <- if (anomaly) runif(1, 1.5, 3.0) else amp
      is_an <- spike > PSI_1 * 1.3 || re_i > 2.5
      strat <- if (re_i > 2.5) "RESET_DELTA" else
               if (e_zone < 0.2) "INCREASE_EPSILON" else "ROLLBACK_LEVEL"
      new_eps <- EPSILON * if (is_an) 1.5 else 1.0

      prompt <- paste0(
        sprintf("H7 executive monitor — anomaly detection at L%d.\n", k),
        sprintf("Expected amplitude range: [%.3f, %.3f]\n",
                PSI_1 * 0.7, PSI_1 * 1.3),
        sprintf("Observed amplitude: %.4f\n", spike),
        sprintf("Re_I: %.3f (%s)\n", re_i,
                if (re_i > 2) "turbulent" else "laminar"),
        sprintf("ε-zone density: %.1f%%\n\n", e_zone * 100),
        "Executive decision:\n",
        "1. Is this an anomaly? (YES/NO)\n",
        "2. If yes, strategy: RESET_DELTA / INCREASE_EPSILON / ROLLBACK_LEVEL\n",
        sprintf("3. Recommended ε adjustment? (current ε = %.4f)\n", EPSILON),
        "Format: anomaly=YES/NO strategy=X new_epsilon=X.XXXX"
      )
      target <- sprintf("anomaly=%s strategy=%s new_epsilon=%.4f",
                        if (is_an) "YES" else "NO",
                        if (is_an) strat else "NONE",
                        new_eps)
      diff <- if (is_an) "hard" else "easy"

    } else {  # redirect_cascade
      target_level <- sample((k + 1):6, 1)
      steps        <- target_level - k
      delta_seq    <- round(k:target_level * DRIFT_072, 4)
      est_amp      <- amp * lambda ^ steps

      prompt <- paste0(
        "H7 cascade redirection task.\n",
        sprintf("Current level: L%d | Target level: L%d\n", k, target_level),
        sprintf("Current amplitude: %.4f | Target zone: %s\n",
                amp, CONSCIOUSNESS_ZONE[as.character(target_level)]),
        sprintf("\nDRIFT_072 = %.6f\n", DRIFT_072),
        sprintf("φ⁷ compression factor = %.4f\n\n", PHI7),
        "Plan the redirection sequence:\n",
        sprintf("1. How many steps to reach L%d?\n", target_level),
        "2. List the δ values for each intermediate level.\n",
        sprintf("3. Estimated amplitude at L%d ", target_level),
        sprintf("(use λ=%.4f per step)?\n", lambda),
        "Format: steps=N deltas=[...] est_amplitude=X.XXXX"
      )
      target <- sprintf("steps=%d deltas=[%s] est_amplitude=%.4f",
                        steps,
                        paste(delta_seq, collapse = ", "),
                        est_amp)
      diff <- "medium"
    }

    rows[[i]] <- data.frame(
      id         = sprintf("exec_%04d", i - 1),
      track      = "executive_functions",
      prompt     = prompt,
      target     = target,
      difficulty = diff,
      subtask    = subtask,
      level_k    = k,
      amplitude  = round(amp, 4),
      re_i       = round(re_i, 4),
      anomaly    = anomaly,
      stringsAsFactors = FALSE
    )
  }
  bind_rows(rows)
}


# ══════════════════════════════════════════════════════════════════════════════
# TRACK 5 — SOCIAL COGNITION
# DeepMind: inferir estados mentales de otros agentes, teoría de la mente.
# H7: inferir zona de consciencia de Sistema B desde su firma observable.
# ══════════════════════════════════════════════════════════════════════════════

generate_social <- function(X, enc, n = 300) {
  H_all <- enc$encode(X)
  T_all <- enc$ternary(H_all)
  amps  <- rowMeans(abs(H_all))

  subtasks <- c("infer_state", "predict_intention", "cooperative_signal")

  rows <- vector("list", n)
  for (i in seq_len(n)) {
    idx_a <- sample(seq_len(nrow(X)), 1)
    idx_b <- sample(seq_len(nrow(X)), 1)

    amp_a <- amps[idx_a]
    amp_b <- amps[idx_b]
    T_b   <- T_all[idx_b, ]

    # Inferir zona de B desde su amplitud
    zone_b  <- if (amp_b > 0.65) "Genetic Memory" else
               if (amp_b > 0.40) "Subconscious" else "Conscious"
    level_b <- if (amp_b > 0.65) sample(0:1, 1) else
               if (amp_b > 0.40) sample(2:5, 1) else sample(6:7, 1)
    intent_b <- if (amp_b > PSI_1) "ENCODING" else "DECODING"
    dist_b   <- abs(amp_b - PSI_1)
    pos_b    <- if (amp_b > PSI_1) "SUBSTRATE" else "FIXED_POINT"

    subtask <- subtasks[((i - 1) %% 3) + 1]

    if (subtask == "infer_state") {
      prompt <- paste0(
        "You are System A in a two-agent H7 holographic network.\n",
        sprintf("Your state: amplitude=%.4f, zone=%s\n\n",
                amp_a, if (amp_a > 0.65) "Genetic Memory" else "Subconscious"),
        "You observe System B's ternary output:\n",
        sprintf("T_B = [%s]... (first 32 of 128 trits)\n",
                paste(head(T_b, 32), collapse = ", ")),
        sprintf("B's mean amplitude: %.4f\n\n", amp_b),
        "Using theory of mind:\n",
        "1. What consciousness zone is System B in?\n",
        "   (Genetic Memory / Subconscious / Conscious)\n",
        "2. Is B closer to the physical substrate (high amplitude) or\n",
        sprintf("   the fixed point |Ψ₁|=%.4f (low amplitude)?\n", PSI_1),
        "3. Estimate B's level (L0-L7).\n",
        "Format: zone=ZONE amplitude_position=SUBSTRATE/FIXED_POINT estimated_level=N"
      )
      target <- sprintf("zone=%s amplitude_position=%s estimated_level=%d",
                        zone_b, pos_b, level_b)
      diff <- "medium"

    } else if (subtask == "predict_intention") {
      active_b <- sum(T_b != 0) / length(T_b)
      steps_est <- max(1, min(6, as.integer(7 * (1 - dist_b / 0.6))))

      prompt <- paste0(
        "Multi-agent H7 scenario.\n",
        sprintf("System A (you): amplitude=%.4f\n", amp_a),
        sprintf("System B (observed): amplitude=%.4f\n", amp_b),
        sprintf("B's ternary signature entropy: %.2f%% active trits\n\n",
                active_b * 100),
        "The H7 cascade has two directions:\n",
        sprintf("  ENCODING: amplitude decreasing toward |Ψ₁|=%.4f\n", PSI_1),
        "  DECODING: amplitude increasing from fixed point outward\n\n",
        "Predict System B's intention:\n",
        "1. Is B encoding or decoding?\n",
        "2. How many cascade steps has B completed?\n",
        "3. What is B's convergence status?\n",
        "Format: intention=ENCODING/DECODING steps=N convergence=CONVERGING/DIVERGING"
      )
      target <- sprintf("intention=%s steps=%d convergence=%s",
                        intent_b, steps_est,
                        if (dist_b < 0.05) "CONVERGING" else "DIVERGING")
      diff <- "hard"

    } else {  # cooperative_signal
      coherence <- 1 - abs(amp_a - amp_b)
      coop_delta <- (level_b + 1) * DRIFT_072
      max_coh    <- min(1.0, coherence + 0.1)
      coop_sig   <- round(cos(pi * Z7 * 0.1 + coop_delta), 4)

      prompt <- paste0(
        "Cooperative H7 protocol.\n",
        sprintf("System A amplitude: %.4f (zone: %s)\n",
                amp_a, if (amp_a < 0.5) "Subconscious" else "Genetic Memory"),
        sprintf("System B amplitude: %.4f (zone: %s)\n", amp_b, zone_b),
        sprintf("Current coherence |1 - |amp_A - amp_B||: %.4f\n\n", coherence),
        "To maximize mutual coherence, System A should transmit\n",
        "a signal that brings its amplitude closer to B's.\n\n",
        sprintf("δ_cooperative = (B_level+1) × DRIFT_072 = %.4f\n\n", coop_delta),
        "1. What is the optimal δ for A to use?\n",
        "2. What 7D signal vector should A transmit?\n",
        "3. What coherence is achievable?\n",
        sprintf("(DRIFT_072 = %.6f, B estimated level = %d)\n", DRIFT_072, level_b),
        "Format: optimal_delta=X.XXXX signal=[...] max_coherence=X.XXXX"
      )
      target <- sprintf("optimal_delta=%.4f signal=[%s] max_coherence=%.4f",
                        coop_delta,
                        paste(coop_sig, collapse = ", "),
                        max_coh)
      diff <- "hard"
    }

    rows[[i]] <- data.frame(
      id         = sprintf("soc_%04d", i - 1),
      track      = "social_cognition",
      prompt     = prompt,
      target     = target,
      difficulty = diff,
      subtask    = subtask,
      amp_a      = round(amp_a, 4),
      amp_b      = round(amp_b, 4),
      zone_b     = zone_b,
      intention_b = intent_b,
      stringsAsFactors = FALSE
    )
  }
  bind_rows(rows)
}


# ══════════════════════════════════════════════════════════════════════════════
# BUILD + SAVE (formato estándar Kaggle)
# ══════════════════════════════════════════════════════════════════════════════

build_and_save <- function(output_dir = "h7_kaggle_r") {

  dir.create(output_dir, showWarnings = FALSE, recursive = TRUE)

  cat("══════════════════════════════════════════════════════════\n")
  cat("  H7 AGI Benchmark  ·  DeepMind 5-Track Alignment (R)\n")
  cat(sprintf("  φ      = %.10f\n", PHI))
  cat(sprintf("  |Ψ₁|   = %.10f\n", PSI_1))
  cat(sprintf("  DRIFT  = %.10f\n", DRIFT_072))
  cat(sprintf("  ε      = %.10f\n", EPSILON))
  cat("══════════════════════════════════════════════════════════\n\n")

  # ── Datos de entrenamiento ────────────────────────────────────────────────
  set.seed(42)
  N <- 300; F <- 7
  X_clean <- matrix(rnorm(N * F), N, F)
  X_noisy <- matrix(rnorm(N/2 * F, sd = 2.5), N/2, F)

  enc <- H7Encoder$new()
  enc$fit(X_clean)

  # ── Generar los 5 tracks ──────────────────────────────────────────────────
  cat("[1] learning             ... "); flush.console()
  df1 <- generate_learning(300)
  cat(sprintf("%d rows\n", nrow(df1)))

  cat("[2] metacognition        ... "); flush.console()
  df2 <- generate_metacognition(X_clean[1:150,], X_noisy[1:150,], enc)
  cat(sprintf("%d rows\n", nrow(df2)))

  cat("[3] attention            ... "); flush.console()
  df3 <- generate_attention(X_clean[1:200,], enc)
  cat(sprintf("%d rows\n", nrow(df3)))

  cat("[4] executive_functions  ... "); flush.console()
  df4 <- generate_executive(300)
  cat(sprintf("%d rows\n", nrow(df4)))

  cat("[5] social_cognition     ... "); flush.console()
  df5 <- generate_social(X_clean[1:150,], enc, 300)
  cat(sprintf("%d rows\n", nrow(df5)))

  # ── Unificar columnas ─────────────────────────────────────────────────────
  keep   <- c("id", "track", "prompt", "target", "difficulty")
  extras <- c("domain", "phi_i", "phi_j", "level_k", "n_index",
              "amplitude", "confidence", "psi1",
              "epsilon_density", "active_trits", "epsilon",
              "subtask", "re_i", "anomaly",
              "amp_a", "amp_b", "zone_b", "intention_b")

  dfs <- list(df1, df2, df3, df4, df5)
  dfs <- lapply(dfs, function(df) {
    for (col in extras) {
      if (!col %in% names(df)) df[[col]] <- NA
    }
    df
  })

  full <- bind_rows(dfs)
  full <- full[, c(keep, intersect(extras, names(full)))]

  # ── Split 80/20 estratificado por track ───────────────────────────────────
  set.seed(42)
  train_idx <- unlist(lapply(unique(full$track), function(tr) {
    sub_idx <- which(full$track == tr)
    sub_idx[sample(seq_along(sub_idx),
                   floor(length(sub_idx) * 0.8))]
  }))
  train_idx <- sort(train_idx)
  test_idx  <- setdiff(seq_len(nrow(full)), train_idx)

  train <- full[train_idx, ]
  test  <- full[test_idx,  ]

  # Test sin target ni columnas de respuesta
  drop_cols  <- c("target", "zone_b", "intention_b")
  test_pub   <- test[, setdiff(names(test), drop_cols)]

  # Sample submission
  sample_sub <- data.frame(
    id     = test$id,
    target = ifelse(test$track %in% c("learning", "metacognition"),
                    "0.00000000",
                    "CONTINUE"),
    stringsAsFactors = FALSE
  )

  # ── Guardar ───────────────────────────────────────────────────────────────
  write.csv(train,     file.path(output_dir, "train.csv"),              row.names = FALSE)
  write.csv(test_pub,  file.path(output_dir, "test.csv"),               row.names = FALSE)
  write.csv(sample_sub,file.path(output_dir, "sample_submission.csv"),  row.names = FALSE)
  write.csv(test,      file.path(output_dir, "test_with_answers.csv"),  row.names = FALSE)

  # ── Resumen ───────────────────────────────────────────────────────────────
  cat(sprintf("\n  Total:  %d rows\n", nrow(full)))
  cat(sprintf("  Train:  %d rows\n", nrow(train)))
  cat(sprintf("  Test:   %d rows\n\n", nrow(test)))

  cat("── Track distribution ──\n")
  dist <- full %>%
    count(track, difficulty) %>%
    arrange(track, difficulty)
  print(as.data.frame(dist), row.names = FALSE)

  # ── Verificación cruzada con Python ──────────────────────────────────────
  cat("\n── Cross-language verification ──\n")
  cat(sprintf("  O(φ¹, φ², n=42, δ=DRIFT_072) = %.10f\n",
              cos(pi * PHI^1 * 42 + DRIFT_072) *
              cos(pi * PHI^2 * 42 - DRIFT_072)))
  cat("  (Python debe dar el mismo valor exacto)\n")
  cat(sprintf("  |Ψ₁| = %.10f\n", PSI_1))
  cat(sprintf("  φ⁷   = %.10f\n", PHI7))

  cat(sprintf("\n[H7-R] Saved → %s/\n", output_dir))
  invisible(list(train = train, test = test_pub, sample = sample_sub))
}


# ══════════════════════════════════════════════════════════════════════════════
# EJECUTAR
# ══════════════════════════════════════════════════════════════════════════════
result <- build_and_save("h7_kaggle_r")

══════════════════════════════════════════════════════════
  H7 AGI Benchmark  ·  DeepMind 5-Track Alignment (R)
  φ      = 1.6180339887
  |Ψ₁|   = 0.3623748901
  DRIFT  = 0.7168146928
  ε      = 0.1811874450
══════════════════════════════════════════════════════════

[1] learning             ... 

300 rows
[2] metacognition        ... 

300 rows
[3] attention            ... 

200 rows
[4] executive_functions  ... 

300 rows
[5] social_cognition     ... 

300 rows

  Total:  1400 rows
  Train:  1120 rows
  Test:   280 rows

── Track distribution ──
               track difficulty   n
           attention       easy 199
           attention     medium   1
 executive_functions       easy  85
 executive_functions       hard 115
 executive_functions     medium 100
            learning       easy  40
            learning       hard 127
            learning     medium 133
       metacognition       easy 150
       metacognition       hard 150
    social_cognition       hard 200
    social_cognition     medium 100

── Cross-language verification ──
  O(φ¹, φ², n=42, δ=DRIFT_072) = 0.5505871900
  (Python debe dar el mismo valor exacto)
  |Ψ₁| = 0.3623748901
  φ⁷   = 29.0344418537

[H7-R] Saved → h7_kaggle_r/


---

## Dataset Card: H7 Cognitive Benchmark for AGI Evaluation

### Overview

Current frontier AI models are highly optimized for token prediction and 
statistical memorization. Evaluating true AGI requires measuring a model's 
capacity for **structural comprehension** — understanding *why* an answer 
is correct, not just recognizing it.

This dataset operationalizes Google DeepMind's cognitive framework 
(*Measuring Progress Toward AGI*) through the **H7 Holographic Data System**: 
a computing architecture derived from a single mathematical axiom.
By projecting information onto a quasi-orthogonal Z₇ basis, we eliminate 
the model's ability to rely on memorized patterns. Every ground truth is 
**mathematically verifiable from first principles**.

---

### 1. System Constants — The Only Axiom

The entire benchmark is derived from **φ = (1+√5)/2**.
No external data, crowdsourced labels, or web-scraped content was used.

| Constant | Value | Role |
|:---|:---:|:---|
| φ (golden ratio) | 1.6180339887 | The only axiom. Irrational basis guaranteeing quasi-orthogonal, non-repeating signatures |
| \|Ψ₁\| (fixed point) | 0.3623748901 | \|cos(π·φ)\| — the holographic attractor every cascade converges to |
| DRIFT\_072 | 0.7168146928 | 7 − 2π — phase offset per level, breaks initial symmetry |
| ε (epsilon threshold) | 0.1811874450 | \|Ψ₁\|/2 — signals below this collapse into the holographic vacuum |
| φ⁷ (Z₇ compression) | 29.034441854 | Compression factor per level: 88B neurons → 147 states in 7 steps |
| C(7,3) | 35 = 7 × 5 | Independent holographic projections in Z₇ |

---

### 2. Dataset Architecture & Cognitive Alignment

**1,400 samples · 5 tracks · 80/20 split (1,120 train / 280 test)**  
Stratified by track and difficulty. Every target is derivable from φ.

| Track | Rows | Task | Metric | Difficulty |
|:---|:---:|:---|:---:|:---:|
| **Learning** | 300 | Transfer O\_{i,j} operator to unseen domains (audio, finance, temperature) via few-shot examples | \|pred − true\| < 0.01 | Easy / Medium / Hard |
| **Metacognition** | 300 | Assess structural integrity: predict deviation from \|Ψ₁\| and classify confidence (HIGH/MEDIUM/LOW) | MAE < 0.02 | Easy / Hard |
| **Attention** | 200 | Reconstruct 7D phase state from 128-trit ternary signature, ignoring ε-vacuum zeros (structured noise) | cosine > 0.85 | Easy / Medium |
| **Executive Functions** | 300 | Plan cascade steps, detect and inhibit anomalies, redirect information flow to target level | Format + numeric accuracy | Easy / Medium / Hard |
| **Social Cognition** | 300 | Infer internal state of a second H7 agent from its observable ternary output (theory of mind) | Zone accuracy > 0.75 | Medium / Hard |

---

### 3. Three-Layer Consciousness Architecture

The benchmark reflects an 88-billion neuron hierarchy compressed via φ⁷:
```
GENETIC MEMORY   L0–L1  88B → 3B     read-only substrate
SUBCONSCIOUS     L2–L5  104M → 4.3K  holographic processing  ← Attention operates here
CONSCIOUS        L6–L7  147 → |Ψ₁|   observer + fixed point
```

---

### 4. Scientific Rigor — Cross-Language Determinism

A benchmark for AGI must be **universally reproducible and platform-agnostic**.  
The H7 operator evaluated at reference parameters:
```
O(φ¹, φ², n=42, δ=DRIFT_072) = 0.5505871900
```

yields the **exact same result in Python, R, and any IEEE 754-compliant language**.  
This was verified independently in both implementations before publication.  
Ground truths cannot be faked, memorized, or language-dependent.

---

### 5. Organizational Affiliation

**smokApp Quantum & AI Independent Research Laboratory**  
Tlaxcala, México · [github.com/jakobmina/H7](https://github.com/jakobmina/H7)
